# 23CSE301 — Machine Learning Capstone — Review 1
## Regression Track — Battery State-of-Health

**Target:** predict **State-of-Health (SOH, %)**
of a battery cell — not raw Capacity directly. Source: NASA Battery Dataset
(Kaggle, `patrickfleith/nasa-battery-dataset`, `cleaned_dataset`) —
`metadata.csv` plus the ~7,000 per-cycle raw CSVs it references via its
`filename` column.

> **Reproducibility:** `random_state=42` throughout (§7.1).


## 0. Imports & configuration

In [3]:
from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold,
    GridSearchCV,
    cross_val_score
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
    PolynomialFeatures
)

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

In [5]:
import warnings
warnings.filterwarnings("ignore")

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold,
    GridSearchCV,
    cross_val_score
)

from sklearn.preprocessing import StandardScaler, PolynomialFeatures

from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error
)

from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.autolayout"] = True

# Your actual dataset location
DATASET_DIR = r"D:\5th Sem\ML\archive (1)\cleaned_dataset"

# Correct absolute paths for your laptop
DATA_PATH = os.path.join(DATASET_DIR, "metadata.csv")
RAW_DIR = os.path.join(DATASET_DIR, "data")

NOMINAL_CAPACITY = 2.0

print("Dataset folder:", DATASET_DIR)
print("Metadata path:", DATA_PATH)
print("Raw data folder:", RAW_DIR)

if os.path.exists(DATA_PATH):
    print("metadata.csv found!")

if os.path.exists(RAW_DIR):
    print("data folder found!")

Dataset folder: D:\5th Sem\ML\archive (1)\cleaned_dataset
Metadata path: D:\5th Sem\ML\archive (1)\cleaned_dataset\metadata.csv
Raw data folder: D:\5th Sem\ML\archive (1)\cleaned_dataset\data
metadata.csv found!
data folder found!


## 1. Load & audit the raw file (Section A1)

In [6]:
df_raw = pd.read_csv(DATA_PATH)
for col in ["Capacity", "Re", "Rct"]:
    df_raw[col] = pd.to_numeric(df_raw[col], errors="coerce")

print("Shape:", df_raw.shape)
print("\nRow type counts:\n", df_raw["type"].value_counts())
print("\nUnique batteries:", df_raw["battery_id"].nunique())
print("\nMissing values per column:\n", df_raw.isna().sum())

Shape: (7565, 10)

Row type counts:
 type
charge       2815
discharge    2794
impedance    1956
Name: count, dtype: int64

Unique batteries: 34

Missing values per column:
 type                      0
start_time                0
ambient_temperature       0
battery_id                0
test_id                   0
uid                       0
filename                  0
Capacity               4796
Re                     5618
Rct                    5618
dtype: int64


**Observation (A1):** 

as before, missingness is structural :-
- `Capacity` only on `discharge` rows, `Re`/`Rct` only on `impedance` rows. 
- `charge` rows carry none of these but do have their own raw per-cycle curve.

### 1b. Structural join: one row per discharge cycle

In [7]:
def parse_start_time(s):
    nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(s))
    if len(nums) < 6:
        return pd.NaT
    y, mo, d, h, mi, se = (float(n) for n in nums[:6])
    try:
        return pd.Timestamp(int(y), int(mo), int(d), int(h), int(mi), int(round(se)))
    except (ValueError, OverflowError):
        return pd.NaT

df_raw["parsed_time"] = df_raw["start_time"].apply(parse_start_time)

discharge = df_raw[df_raw["type"] == "discharge"].sort_values(["battery_id", "test_id"]).copy()
impedance = df_raw[df_raw["type"] == "impedance"].sort_values(["battery_id", "test_id"]).copy()
charge = df_raw[df_raw["type"] == "charge"].sort_values(["battery_id", "test_id"]).copy()

discharge["cycle_index"] = discharge.groupby("battery_id").cumcount() + 1

disc_slim = discharge[["battery_id", "test_id", "ambient_temperature", "cycle_index", "Capacity", "filename"]]
imp_slim = impedance[["battery_id", "test_id", "Re", "Rct"]]

analysis_df = pd.merge_asof(
    disc_slim.sort_values("test_id"), imp_slim.sort_values("test_id"),
    on="test_id", by="battery_id", direction="backward",
)
print("Per-discharge-cycle table:", analysis_df.shape)
analysis_df.head()


Per-discharge-cycle table: (2794, 8)


,battery_id,test_id,ambient_temperature,cycle_index,Capacity,filename,Re,Rct
0,B0053,0,4,1,1.069142,06672.csv,NaN,NaN
1,B0033,0,24,1,0.068426,02413.csv,NaN,NaN
2,B0052,0,4,1,0.860659,04381.csv,NaN,NaN
3,B0055,0,4,1,0.799000,07314.csv,NaN,NaN
4,B0051,0,4,1,0.643474,04443.csv,NaN,NaN


## 2. Feature extraction from the raw per-cycle CSVs


Each `discharge`/`charge` row's `filename` points to a raw curve with columns
`Voltage_measured, Current_measured, Temperature_measured, Current_load/charge,
Voltage_load/charge, Time`.

**Why not just use the full curve?** 

Checking one discharge file end to end: the
load current holds at ≈-0.995 A until cutoff at 2.47 V, `Time≈6179s` — working
out `0.995A × (6179s/3600) ≈ 1.71 Ah` lines up almost exactly with that row's
`Capacity = 1.674 Ah`. That's expected, since Capacity **is defined as** the
discharge current integrated over time — but it means a feature built from the
**full** discharge curve (total duration, full-curve integral) is just the
target restated, not a real predictor. We deliberately avoid that:

- **From each discharge cycle's own curve** — only the **first ~300 seconds**:
  voltage-sag rate and temperature-rise rate under load. This reflects internal
  resistance, not how long the discharge ran.
- **From the *preceding* charge cycle** (nearest-prior per battery, same join
  pattern as `Re`/`Rct`): full charge duration, max temperature, and time to
  reach the CC→CV voltage threshold. This is a separate physical process, so
  using its full curve doesn't restate this discharge's capacity.

In [8]:
def extract_discharge_early_features(path, early_seconds=300):
    d = pd.read_csv(path, usecols=["Voltage_measured", "Temperature_measured", "Time"])
    v0, t0 = d["Voltage_measured"].iloc[0], d["Temperature_measured"].iloc[0]
    window = min(early_seconds, 0.2 * d["Time"].max())
    early = d[d["Time"] <= window]
    if len(early) < 2:
        early = d.iloc[:max(2, int(0.1 * len(d)))]
    v_end, t_end, temp_end = early["Voltage_measured"].iloc[-1], early["Time"].iloc[-1], early["Temperature_measured"].iloc[-1]
    return {
        "early_voltage_sag_rate": (v0 - v_end) / t_end if t_end > 0 else np.nan,
        "early_temp_rise_rate": (temp_end - t0) / t_end if t_end > 0 else np.nan,
    }

def extract_charge_features(path, cv_threshold=4.15):
    d = pd.read_csv(path, usecols=["Voltage_measured", "Temperature_measured", "Time"])
    duration, max_temp = d["Time"].max(), d["Temperature_measured"].max()
    above = d[d["Voltage_measured"] >= cv_threshold]
    time_to_cv = above["Time"].iloc[0] if len(above) else duration
    return {"charge_duration": duration, "charge_max_temp": max_temp, "charge_time_to_cv": time_to_cv}


In [ ]:
disc_feats = []
for _, row in discharge.iterrows():
    try:
        feats = extract_discharge_early_features(f"{RAW_DIR}/{row['filename']}")
    except (FileNotFoundError, pd.errors.EmptyDataError, KeyError):
        feats = {"early_voltage_sag_rate": np.nan, "early_temp_rise_rate": np.nan}
    feats["filename"] = row["filename"]
    disc_feats.append(feats)
disc_feats_df = pd.DataFrame(disc_feats)

chg_feats = []
for _, row in charge.iterrows():
    try:
        feats = extract_charge_features(f"{RAW_DIR}/{row['filename']}")
    except (FileNotFoundError, pd.errors.EmptyDataError, KeyError):
        feats = {"charge_duration": np.nan, "charge_max_temp": np.nan, "charge_time_to_cv": np.nan}
    feats["battery_id"], feats["test_id"] = row["battery_id"], row["test_id"]
    chg_feats.append(feats)
chg_feats_df = pd.DataFrame(chg_feats).sort_values("test_id")

print("Discharge-curve features:", disc_feats_df.shape, " | missing:\n", disc_feats_df.isna().sum().to_dict())
print("Charge-curve features:", chg_feats_df.shape, " | missing:\n", chg_feats_df.isna().sum().to_dict())

# Merge own discharge-curve features (direct join, same file) ...
analysis_df = analysis_df.merge(disc_feats_df, on="filename", how="left")
# ... and the nearest-preceding charge cycle's features (asof join, like Re/Rct)
analysis_df = pd.merge_asof(
    analysis_df.sort_values("test_id"), chg_feats_df,
    on="test_id", by="battery_id", direction="backward",
)
print("\nEnriched analysis table:", analysis_df.shape)
analysis_df.head()


## 3. Exploratory Data Analysis (Section A2)

In [ ]:
battery_summary = analysis_df.groupby("battery_id")["Capacity"].agg(
    n_cycles="size", missing_frac=lambda s: s.isna().mean(), min="min", max="max"
).sort_values("n_cycles")
battery_summary


In [ ]:
raw_feats = ["early_voltage_sag_rate", "early_temp_rise_rate", "charge_duration", "charge_max_temp", "charge_time_to_cv"]
fig, axes = plt.subplots(1, len(raw_feats), figsize=(4 * len(raw_feats), 3.6))
for ax, col in zip(axes, raw_feats):
    sns.histplot(analysis_df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(col, fontsize=10)
plt.suptitle("A2 — New raw-curve feature distributions (pre-cleaning)", y=1.05)
plt.savefig("a2_raw_feature_distributions.png", bbox_inches="tight")
plt.show()


In [ ]:
corr_cols = ["ambient_temperature", "cycle_index", "Capacity", "Re", "Rct"] + raw_feats
plt.figure(figsize=(8, 6.5))
sns.heatmap(analysis_df[corr_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, square=True)
plt.title("A2 — Correlation heatmap (incl. new raw-curve features)")
plt.savefig("a2_correlation_heatmap.png", bbox_inches="tight")
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for bid, g in analysis_df.groupby("battery_id"):
    axes[0].plot(g["cycle_index"], g["Capacity"], marker=".", linewidth=0.7, alpha=0.6)
axes[0].set_xlabel("Cycle index"); axes[0].set_ylabel("Capacity (Ah)")
axes[0].set_title("A2 — Capacity fade vs. cycle (every battery)")

axes[1].scatter(analysis_df["early_voltage_sag_rate"], analysis_df["Capacity"], alpha=0.3, s=12, color="teal")
axes[1].set_xlabel("Early voltage-sag rate (V/s)"); axes[1].set_ylabel("Capacity (Ah)")
axes[1].set_title("A2 — Capacity vs. early voltage-sag rate")
plt.savefig("a2_scatterplots.png", bbox_inches="tight")
plt.show()


**Observation (A2):**
- The fade curves reproduce the same pattern seen before: a curved decline per
  battery, plus the same four short, anomalous runs.
- **B0049, B0050, B0051, B0052** again stand out in the per-battery summary —
  25 cycles each vs. 28–197 for every other battery, and B0052 alone is
  missing 84% of its Capacity readings. That gap in cycle count and
  missingness, on its own, is reason enough to treat these four as unreliable
  rather than as genuinely short-lived cells. A separate, consolidated
  mirror of this same NASA PCoE dataset on Kaggle ("Li-ion Battery Dataset
  from NASA PCoE", ckskaggle) groups these same four IDs together (Group G8,
  4°C, 2A CC) and marks all four of them "crashed" in its metadata summary —
  consistent with, though not a primary NASA source for, what the EDA above
  already shows directly. Excluded in Section 5 (B1) on the strength of our
  own evidence, same as before.
- The new `early_voltage_sag_rate` feature shows a visible negative
  relationship with Capacity — a faster voltage sag under load early in
  discharge tracks a more degraded (higher-impedance) cell, as hoped.

## 4. Data cleaning (Section B1)

In [ ]:
CORRUPTED_BATTERIES = ["B0049", "B0050", "B0051", "B0052"]
df_clean = analysis_df[~analysis_df["battery_id"].isin(CORRUPTED_BATTERIES)].copy()
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
print(f"Rows after dropping corrupted-log batteries & duplicates: {df_clean.shape[0]}")

fill_cols = ["Re", "Rct"] + raw_feats
for col in fill_cols:
    df_clean[col] = df_clean.groupby("battery_id")[col].transform(lambda s: s.bfill().ffill())
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())
print("Remaining missing values:\n", df_clean[fill_cols + ["Capacity"]].isna().sum())

q1, q3 = df_clean["Capacity"].quantile([0.25, 0.75]); iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
n_outliers = ((df_clean["Capacity"] < lower) | (df_clean["Capacity"] > upper)).sum()
df_clean["Capacity"] = df_clean["Capacity"].clip(lower, upper)
print(f"\nCapacity outliers outside [{lower:.3f}, {upper:.3f}]: {n_outliers} (capped, not dropped)")


**Justification (B1):** same reasoning as before — 
- corrupted-log batteries dropped outright (too little reliable signal to salvage), remaining
gaps in impedance/curve-derived features filled from each battery's **own**
nearest reading, and Capacity outliers **capped** rather than dropped so
`cycle_index` sequences stay intact for every remaining battery.

## 5. Target engineering — SOH

**TARGET:** the regression target is **State-of-Health (SOH, %)**,
not raw Capacity. `SOH_pct = 100 × Capacity / NOMINAL_CAPACITY`, defined for
every row.


In [ ]:
df_clean["SOH_pct"] = 100 * df_clean["Capacity"] / NOMINAL_CAPACITY

print(f"SOH defined for all {df_clean.shape[0]} rows across {df_clean['battery_id'].nunique()} batteries.")
print("\nSOH_pct:\n", df_clean["SOH_pct"].describe())

## 6. Feature set, and why we don't use a random row-level split (Section B2)

**This is panel data** — repeated cycles from the same ~30 batteries, not
independent rows — and `battery_id` is a candidate feature. If the *same*
battery's cycles are scattered across both train and test (a random row-level
split), the model can partly fit battery-specific quirks via that battery's
one-hot column instead of learning the general relationship between the
physical features and SOH. Quick comparison, run once:


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

_numeric_demo = ["ambient_temperature", "Re", "Rct", "cycle_index"] + raw_feats

# (a) row-level split, battery_id included as a feature
_X = df_clean[_numeric_demo + ["battery_id"]]
_y = df_clean["SOH_pct"]
_Xtr, _Xte, _ytr, _yte = train_test_split(_X, _y, test_size=0.2, random_state=RANDOM_STATE)
_pre = ColumnTransformer([("num", StandardScaler(), _numeric_demo),
                           ("cat", OneHotEncoder(handle_unknown="ignore"), ["battery_id"])],
                          sparse_threshold=0).fit(_Xtr)
_row_level_r2 = r2_score(_yte, LinearRegression().fit(_pre.transform(_Xtr), _ytr).predict(_pre.transform(_Xte)))
print(f"(a) Row-level split + battery_id feature       -> Test R2 = {_row_level_r2:.4f}")

# (b) battery-grouped split, battery_id dropped
_groups = df_clean["battery_id"]
_gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
_tr_idx, _te_idx = next(_gss.split(df_clean[_numeric_demo], _y, _groups))
_sc = StandardScaler().fit(df_clean[_numeric_demo].iloc[_tr_idx])
_grouped_r2 = r2_score(
    _y.iloc[_te_idx],
    LinearRegression().fit(_sc.transform(df_clean[_numeric_demo].iloc[_tr_idx]), _y.iloc[_tr_idx])
      .predict(_sc.transform(df_clean[_numeric_demo].iloc[_te_idx]))
)
print(f"(b) Battery-grouped split, no battery_id feature -> Test R2 = {_grouped_r2:.4f}")


(a) scores noticeably higher than (b) — the same effect a fully
memorisable target (like remaining-life-to-a-fixed-cycle) would show far more
dramatically, just milder here since SOH still requires learning an actual
degradation shape rather than recalling one constant. Either way, (a) is
answering an easier question than the one we actually care about ("how well
does this work on a battery the model has never seen?"). 

**The fix:** split by **battery**, not by row — every cycle from a given battery goes entirely
into train *or* test, never both — and **drop `battery_id` as a model
feature**, since a battery-level split makes it structurally unable to
generalise to unseen batteries anyway (its one-hot column is all zeros for
every test battery). This also simplifies the pipeline: no more categorical
encoding, just `StandardScaler` on the physical features.

In [ ]:
FEATURE_COLS = ["ambient_temperature", "Re", "Rct", "cycle_index"] + raw_feats
print("Final feature set:", FEATURE_COLS)


## 7. Train/test split & scaling (Section B2, applied)

`GroupShuffleSplit` — every cycle from a given battery goes entirely into
train *or* test, never both — 80:20 by battery, `battery_id` not used as a
feature (see Section 6). `StandardScaler` is fit on the training split only.


In [ ]:
X = df_clean[FEATURE_COLS].copy()
y = df_clean["SOH_pct"].copy()
groups = df_clean["battery_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
groups_train = groups.iloc[train_idx]

print(f"Train: {len(X_train)} rows / {groups_train.nunique()} batteries")
print(f"Test:  {len(X_test)} rows / {groups.iloc[test_idx].nunique()} batteries")
print("Battery overlap between train/test:", set(groups_train) & set(groups.iloc[test_idx]))


In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print("Scaled feature matrix shapes:", X_train_s.shape, X_test_s.shape)


## 8. Feature engineering (Section B3)

`cycle_index_squared` — Capacity/SOH fades in a curve, not a line, so squaring
the (already-scaled) cycle index lets the linear-family models below capture
that curvature. Computed once here, after scaling, and reused by every
algorithm that isn't Polynomial Regression (which generates its own
higher-order terms directly).


In [ ]:
cyc_pos = FEATURE_COLS.index("cycle_index")
X_train_fe = np.hstack([X_train_s, (X_train_s[:, [cyc_pos]]) ** 2])
X_test_fe = np.hstack([X_test_s, (X_test_s[:, [cyc_pos]]) ** 2])
feature_names_fe = FEATURE_COLS + ["cycle_index_squared"]
print("Final engineered feature count:", X_train_fe.shape[1])


## 9. Regression Track, target = SOH (Section C)

For each algorithm: train → predict on the held-out (unseen-battery) test set
→ compute **R², RMSE, MAE** (all mandatory). Results are collected into
`results` for the Section C2 comparison table; `model_lookup` keeps each
fitted model + the exact feature matrix it was trained on, for the CV step
and the C4 plots later.


In [ ]:
results = {}
model_lookup = {}

def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    preds = model.predict(X_te)
    r2 = r2_score(y_te, preds)
    rmse = np.sqrt(mean_squared_error(y_te, preds))
    mae = mean_absolute_error(y_te, preds)
    results[name] = {"R2": r2, "RMSE": rmse, "MAE": mae}
    model_lookup[name] = (model, X_tr)
    print(f"{name:32s} | R2={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}")
    return model


### C1.1 — Linear Regression (baseline)

In [ ]:
lin_model = evaluate(
    "1. Linear Regression", LinearRegression(),
    X_train_fe, y_train, X_test_fe, y_test,
)

coefs = pd.Series(lin_model.coef_, index=feature_names_fe).sort_values(key=np.abs, ascending=False)
print("\nCoefficient magnitudes (standardised features, so directly comparable):")
print(coefs)


In [ ]:
# Sanity check: does each feature's coefficient sign in the JOINT model match
# its own simple correlation with the target? A mismatch is the signature of
# multicollinearity — the model isn't "wrong", but that feature's coefficient
# can't be read as an isolated effect.
simple_corr = df_clean[FEATURE_COLS + ["SOH_pct"]].corr()["SOH_pct"].drop("SOH_pct")
check = pd.DataFrame({
    "joint_coef": pd.Series(lin_model.coef_[:len(FEATURE_COLS)], index=FEATURE_COLS),
    "simple_corr_with_SOH": simple_corr,
})
check["sign_flip"] = np.sign(check["joint_coef"]) != np.sign(check["simple_corr_with_SOH"])
print(check.sort_values("joint_coef", key=np.abs, ascending=False))

print("\nMost collinear feature pairs (|r| > 0.5):")
fcorr = df_clean[FEATURE_COLS].corr()
for i in range(len(FEATURE_COLS)):
    for j in range(i + 1, len(FEATURE_COLS)):
        r = fcorr.iloc[i, j]
        if abs(r) > 0.5:
            print(f"  {FEATURE_COLS[i]:22s} <-> {FEATURE_COLS[j]:22s}  r={r:.2f}")


**Interpretation:** 
`cycle_index`/`cycle_index_squared` turn out to have
the *smallest* coefficients, not the largest — `charge_time_to_cv`,
`charge_max_temp`, and `ambient_temperature` dominate instead. The check above
shows why that's not simply "cycle count doesn't matter": `ambient_temperature`
correlates 0.94 with `charge_max_temp`, 0.82 with `charge_time_to_cv`, and
‑0.78 with `Rct` — because NASA's test protocol fixes a matching charge/discharge
regime per ambient-temperature group, so these four features are largely
standing in for **which temperature group a battery belongs to**, not four
independent physical signals. That collinearity also **flips coefficient
signs** for `charge_max_temp`, `Re`, and `early_voltage_sag_rate` relative to
their own simple correlation with SOH (flagged `True` above) — a classic
multicollinearity symptom, not a real reversal of the physical relationship.
`Rct` is the one feature whose sign agrees both alone (r=‑0.71) and in the
joint model — the most trustworthy single coefficient here. Reducing the
temperature-group redundancy (e.g. keeping only one of `ambient_temperature` /
`charge_max_temp` / `charge_time_to_cv`) would make these coefficients more
individually interpretable, if there's time to revisit it.

### C1.2 — Ridge Regression (L2, tuned alpha)

In [ ]:
ridge_grid = GridSearchCV(
    Ridge(random_state=RANDOM_STATE),
    param_grid={"alpha": [0.001, 0.01, 0.1, 1, 10, 100]},
    scoring="r2", cv=5, n_jobs=-1,
)
ridge_grid.fit(X_train_fe, y_train)
print("Best Ridge alpha:", ridge_grid.best_params_, " | CV R2:", round(ridge_grid.best_score_, 4))

ridge_model = evaluate(
    "2. Ridge Regression", Ridge(alpha=ridge_grid.best_params_["alpha"], random_state=RANDOM_STATE),
    X_train_fe, y_train, X_test_fe, y_test,
)


### C1.3 — Lasso Regression (L1, tuned alpha, sparsity check)

In [ ]:
lasso_grid = GridSearchCV(
    Lasso(random_state=RANDOM_STATE, max_iter=20000),
    param_grid={"alpha": [0.0001, 0.001, 0.01, 0.05, 0.1, 0.5]},
    scoring="r2", cv=5, n_jobs=-1,
)
lasso_grid.fit(X_train_fe, y_train)
print("Best Lasso alpha:", lasso_grid.best_params_, " | CV R2:", round(lasso_grid.best_score_, 4))

lasso_model = evaluate(
    "3. Lasso Regression", Lasso(alpha=lasso_grid.best_params_["alpha"], random_state=RANDOM_STATE, max_iter=20000),
    X_train_fe, y_train, X_test_fe, y_test,
)

n_zeroed = int((np.abs(lasso_model.coef_) < 1e-8).sum())
print(f"\nLasso zeroed {n_zeroed}/{len(lasso_model.coef_)} coefficients (L1 sparsity).")


### C1.4 — ElasticNet Regression (combined L1+L2, tuned alpha & l1_ratio)

In [ ]:
enet_grid = GridSearchCV(
    ElasticNet(random_state=RANDOM_STATE, max_iter=20000),
    param_grid={"alpha": [0.0001, 0.001, 0.01, 0.1], "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]},
    scoring="r2", cv=5, n_jobs=-1,
)
enet_grid.fit(X_train_fe, y_train)
print("Best ElasticNet params:", enet_grid.best_params_, " | CV R2:", round(enet_grid.best_score_, 4))

enet_model = evaluate(
    "4. ElasticNet Regression",
    ElasticNet(**enet_grid.best_params_, random_state=RANDOM_STATE, max_iter=20000),
    X_train_fe, y_train, X_test_fe, y_test,
)


### C1.5 — Polynomial Regression (PolynomialFeatures + Linear Regression, compare degrees)

In [ ]:
# Fit on the pre-engineering scaled features (X_*_s): PolynomialFeatures
# generates its own higher-order terms, so stacking it on the already-squared
# cycle_index_fe set would duplicate that information.
poly_scores = {}
for degree in [2, 3]:
    poly = PolynomialFeatures(degree=degree, include_bias=False)
    X_tr_poly = poly.fit_transform(X_train_s)
    X_te_poly = poly.transform(X_test_s)
    m = LinearRegression().fit(X_tr_poly, y_train)
    poly_scores[degree] = r2_score(y_test, m.predict(X_te_poly))
    print(f"Polynomial degree={degree} -> Test R2={poly_scores[degree]:.4f}, n_features={X_tr_poly.shape[1]}")

best_degree = max(poly_scores, key=poly_scores.get)
print(f"\nBest polynomial degree by test R2: {best_degree}")

poly_best = PolynomialFeatures(degree=best_degree, include_bias=False)
X_train_poly = poly_best.fit_transform(X_train_s)
X_test_poly = poly_best.transform(X_test_s)

poly_model = evaluate(
    f"5. Polynomial Regression (deg={best_degree})", LinearRegression(),
    X_train_poly, y_train, X_test_poly, y_test,
)


## 10. Comparative evaluation table (Section C2)

In [ ]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("R2", ascending=False)
results_df.insert(0, "Rank", range(1, len(results_df) + 1))
results_df


## 11. Regression Algorithms 6–10

The following five regression algorithms are trained using the same
preprocessed training and testing datasets:

6. Decision Tree Regressor  
7. Random Forest Regressor  
8. Gradient Boosting Regressor  
9. Support Vector Regressor  
10. K-Nearest Neighbors Regressor  

All models are evaluated using R², RMSE and MAE on the same held-out,
battery-wise test set.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor